In [1]:
!pip install RapidFuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 31.7 MB/s  0:00:00


## Step B — Load wine names from BigQuery

In [2]:
import pandas as pd
import pandas_gbq
import os
from google.oauth2 import service_account

# Get credentials from environment variable
credentials_path = os.environ.get('GOOGLE_APPLICATION_CREDENTIALS')

credentials = service_account.Credentials.from_service_account_file(
    credentials_path
)

In [3]:
query_event = """
SELECT customer_id, company, wine_name, article_code, volume_in_ml, price_in_eur, price_per_100ml, `750_equiv`, discount, timestamp, year, month, day, hour, visit_count
FROM `correct-project-475814.wine_batch_2184.event_table`
WHERE wine_name IS NOT NULL
"""

df_event = pandas_gbq.read_gbq(
    query_event,
    project_id="correct-project-475814",
    credentials=credentials
)

Downloading: 100%|█████████████████████████████████████████████████████████████|


In [5]:
query_xwines = """
SELECT WineID, WineName, Type, Elaborate, Grapes, Harmonize, ABV, Body, Acidity, Country, RegionName, WineryName
FROM `correct-project-475814.wine_batch_2184.x_wines_raw`
WHERE WineName IS NOT NULL
"""


In [7]:


df_xwines = pandas_gbq.read_gbq(
    query_xwines,
    project_id="correct-project-475814",
    credentials=credentials
)

Downloading: 100%|█████████████████████████████████████████████████████████████|


In [9]:
!pip install RapidFuzz

In [ ]:
from rapidfuzz import fuzz, process
import pandas as pd


def normalize_name(x):
    if pd.isna(x): return ""
    return (
        x.lower()
         .replace("'","'")
         .replace("-"," ")
         .strip()
    )

df_event['name_norm'] = df_event['wine_name'].apply(normalize_name)
df_xwines['name_norm']     = df_xwines['WineName'].apply(normalize_name)


choices = df_xwines['name_norm'].tolist()

def best_match(row):
    q = row['name_norm']
    match, score, idx = process.extractOne(q, choices, scorer=fuzz.WRatio)
    return pd.Series([match, score, idx])

df_event[['match_name','score','x_idx']] = df_event.apply(best_match, axis=1)


cols_from_x = [
    'WineID','WineName','Type','Elaborate','Grapes','Harmonize','ABV','Body',
    'Acidity','Code','Country','RegionID','RegionName','WineryID','WineryName',
    'Website','Vintages','match_text','cluster','vintage_count'
]

df_event = df_event.drop(columns=[c for c in cols_from_x if c in df_event.columns], errors='ignore')


df_event = df_event.drop(columns=['name_norm'], errors='ignore')
df_x_clean = df_xwines.drop(columns=['name_norm'], errors='ignore')


df_event = df_event.merge(
    df_x_clean,
    left_on='x_idx',
    right_index=True,
    how='left'
)


df_event['accepted'] = df_event['score'] >= 90

df_event.head()